In [1]:
import pandas as pd
from pathlib import Path

PROJECT_ROOT = Path.cwd()
while not (PROJECT_ROOT / "CLAUDE.md").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

df = pd.read_parquet(
    PROJECT_ROOT / "data" / "raw" / "statcast_2024-04-15_ingested_2026-08-31.parquet"
).sort_values(["game_pk", "at_bat_number", "pitch_number"]).reset_index(drop=True)

print(df.shape)
print(sorted(df["description"].unique()))

(4362, 119)
['automatic_ball', 'ball', 'blocked_ball', 'called_strike', 'foul', 'foul_bunt', 'foul_tip', 'hit_by_pitch', 'hit_into_play', 'missed_bunt', 'swinging_strike', 'swinging_strike_blocked']


In [2]:
SWING_DESCRIPTIONS = {
    "foul",
    "hit_into_play",
    "swinging_strike",
    "swinging_strike_blocked",
    "foul_tip",
}

BUNT_DESCRIPTIONS = {"foul_bunt", "missed_bunt"}

WHIFF_DESCRIPTIONS = {
    "swinging_strike",
    "swinging_strike_blocked",
}

df["is_swing"] = df["description"].isin(SWING_DESCRIPTIONS)
df["is_whiff"] = df["description"].isin(WHIFF_DESCRIPTIONS)
df["is_bunt"] = df["description"].isin(BUNT_DESCRIPTIONS)

print("swings:", df["is_swing"].sum())
print("whiffs:", df["is_whiff"].sum())
print("bunts (excluded):", df["is_bunt"].sum())

swings: 2044
whiffs: 436
bunts (excluded): 9


In [3]:
swings = df["is_swing"].sum()
whiffs = df["is_whiff"].sum()
pitches = len(df)

print(f"Swing%: {swings/pitches:.1%}  ({swings}/{pitches})")
print(f"Whiff%: {whiffs/swings:.1%}  ({whiffs}/{swings})")
print(f"SwStr%: {whiffs/pitches:.1%}  ({whiffs}/{pitches})")

Swing%: 46.9%  (2044/4362)
Whiff%: 21.3%  (436/2044)
SwStr%: 10.0%  (436/4362)


In [4]:
covered = SWING_DESCRIPTIONS | BUNT_DESCRIPTIONS | WHIFF_DESCRIPTIONS
all_desc = set(df["description"].unique())
non_swing = all_desc - covered

print("classified as non-swing:")
for d in sorted(non_swing):
    print(" ", d, (df["description"] == d).sum())

print()
print("whiffs not in swings:", (df["is_whiff"] & ~df["is_swing"]).sum())

classified as non-swing:
  automatic_ball 16
  ball 1436
  blocked_ball 92
  called_strike 758
  hit_by_pitch 7

whiffs not in swings: 0


In [5]:
by_count = (
    df[df["is_swing"]]
    .groupby(["balls", "strikes"])
    .agg(swings=("is_swing", "sum"), whiffs=("is_whiff", "sum"))
)
by_count["whiff_pct"] = by_count["whiffs"] / by_count["swings"]
print(by_count.sort_values("whiff_pct", ascending=False).to_string())

               swings  whiffs  whiff_pct
balls strikes                           
0     2           140      35   0.250000
1     0           176      44   0.250000
0     1           283      70   0.247350
      0           329      80   0.243161
1     2           220      46   0.209091
      1           249      50   0.200803
3     0             5       1   0.200000
2     2           246      47   0.191057
3     1            48       9   0.187500
2     1           126      21   0.166667
3     2           173      27   0.156069
2     0            49       6   0.122449
